# Module 2: LLM Integration
In [Module 1](https://www.kaggle.com/code/anhoangvo/kaggle-benchmarks-core-concepts), we learned how Kaggle Benchmarks records conversations with actors, messages, chats, and contexts.
Now we will connect that conversation record to model-backed actors.
The main new idea is `LLMChat`.
An `LLMChat` is still an actor, so it can send messages into a chat.
The difference is that it can also read the active chat, call a model provider, and write the model's response back into the same chat.
Think of it like a helpful translator at a service desk:
- the chat is the paper form with the conversation history,
- `LLMChat.respond(...)` reads the form,
- the provider produces an answer,
- the answer is stamped back onto the form as a message.
To keep this notebook runnable for everyone, we will use a tiny deterministic `LLMChat` first.
It behaves like a provider, but it does not call the internet or require credentials.
After the mechanics are clear, we will look at the real Kaggle `ModelProxy` setup pattern.

## Source Map
These are the framework files and tests behind this lesson in [Kaggle’s public GitHub repository](https://github.com/Kaggle/kaggle-benchmarks):
- [src/kaggle_benchmarks/actors/llms.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/src/kaggle_benchmarks/actors/llms.py) defines `LLMChat`, `LLMResponse`, `OpenAI`, `GoogleGenAI`, reasoning handling, `.prompt(...)`, and `.respond(...)`.
- [src/kaggle_benchmarks/kaggle/model_proxy.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/src/kaggle_benchmarks/kaggle/model_proxy.py) defines `ModelProxy`, which builds an OpenAI or Google GenAI backed `LLMChat` from Kaggle model proxy settings.
- [src/kaggle_benchmarks/kaggle/models.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/src/kaggle_benchmarks/kaggle/models.py) provides helpers for loading default, judge, and available model proxy models from environment variables.
- [src/kaggle_benchmarks/serializers/openai.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/src/kaggle_benchmarks/serializers/openai.py) converts framework messages into OpenAI-compatible request messages.
- [src/kaggle_benchmarks/serializers/genai.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/src/kaggle_benchmarks/serializers/genai.py) converts framework messages into Google GenAI-compatible request content.
- [tests/test_llm_chats.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/tests/test_llm_chats.py) shows `.prompt(...)`, `.respond(...)`, chat history, system instructions, forks, and usage aggregation.
- [tests/test_openai_client.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/tests/test_openai_client.py) shows OpenAI reasoning, tool call, streaming, and token behavior.
- [tests/test_genai_client.py](https://github.com/Kaggle/kaggle-benchmarks/blob/ci/tests/test_genai_client.py) shows Google GenAI reasoning, token, and multimodal behavior.

## Table of Contents
* [1. Why LLM Integration Matters](#why-llm-integration-matters)
* [2. A Tiny Local LLMChat](#tiny-local-llmchat)
* [3. prompt: Ask In One Step](#prompt)
* [4. respond: Answer From The Active Chat](#respond)
* [5. What Gets Sent To The LLM?](#what-gets-sent)
* [6. System Instructions](#system-instructions)
* [7. Provider Responses And Metadata](#metadata)
* [8. Streaming Responses](#streaming)
* [9. Kaggle ModelProxy](#model-proxy)
* [10. Common Gotchas](#common-gotchas)
* [11. Practice Example](#practice-example)
* [12. Recap](#recap)

## 1. Why LLM Integration Matters <a class="anchor" id="why-llm-integration-matters" name="why-llm-integration-matters">&#8203;</a>
A benchmark usually needs more than a prompt and a final answer.
It needs a record of the prompt, the model response, any hidden setup, and the usage metadata that came back from the model provider.
`LLMChat` gives the framework one shared shape for that work.
Whether the backend is OpenAI-compatible, Google GenAI, or Kaggle Model Proxy, the benchmark code can use the same high-level methods:
- `.prompt(...)` for a one-shot user prompt
- `.respond(...)` for answering from the current chat
- `.send(...)` for normal actor-style messages
This means the provider can change while the chat logic stays familiar.

Let's import the pieces for the lesson.
We will use `LLMResponse` to mimic a provider result.
We will use `LLMChat` to create a small model-backed actor that is safe to run in a notebook.

In [ ]:
import os

# Use DOC mode so Kaggle's rendered notebook output can display benchmark UI
# without relying on live Jupyter widget comms.
#
# Run this cell first before importing kaggle_benchmarks.
# If you rerun this notebook only to see the correct interactive display,
# you can ignore this cell.
os.environ["BENCHMARK_MODE"] = "DOC"

In [ ]:
from kaggle_benchmarks import actors, chats
from kaggle_benchmarks.actors.llms import LLMChat, LLMResponse

## 2. A Tiny Local LLMChat <a class="anchor" id="tiny-local-llmchat" name="tiny-local-llmchat">&#8203;</a>
Real provider classes implement `.invoke(...)` by serializing messages and calling an API.
For learning, we can make a small local class that implements the same contract without a network call.
This keeps the important shape visible:
- `.respond(...)` gathers visible messages from the active chat.
- `.invoke(...)` receives those messages.
- `.invoke(...)` returns an `LLMResponse`.
- `.respond(...)` turns that response into a normal chat message.

In [ ]:
class NotebookLLM(LLMChat):
    """A deterministic LLMChat for learning the framework contract."""

    def __init__(self, name="Notebook LLM"):
        super().__init__(name=name, support_temperature=True)
        self.stream_responses = False
        self.invocations = []

    def invoke(self, messages, system=None, reasoning=None, **kwargs):
        visible_texts = [message.text for message in messages]
        latest = visible_texts[-1] if visible_texts else "(no visible messages)"
        answer = (
            f"I saw {len(messages)} visible message(s). "
            f"Latest visible message: {latest}"
        )

        self.invocations.append(
            {
                "messages": [
                    {
                        "sender": message.sender.name,
                        "role": message.sender.role,
                        "text": message.text,
                    }
                    for message in messages
                ],
                "system": system,
                "reasoning": reasoning,
                "kwargs": kwargs,
            }
        )

        reasoning_traces = None
        if reasoning not in (None, "none"):
            reasoning_traces = "Checked the visible chat messages and summarized the latest one."

        return LLMResponse(
            content=answer,
            reasoning_traces=reasoning_traces,
            meta={
                "input_tokens": sum(len(text.split()) for text in visible_texts),
                "output_tokens": len(answer.split()),
                "total_backend_latency_ms": 7,
            },
        )


agent = NotebookLLM()

print(agent)
print("Role:", agent.role)
print("Is an actor:", isinstance(agent, actors.Actor))

Nice, we have a model-backed actor.
It has the assistant role by default, and it still behaves like an actor.
The provider-specific work is isolated inside `.invoke(...)`.
In a real provider class, `.invoke(...)` is where the API request happens.

## 3. prompt: Ask In One Step <a class="anchor" id="prompt" name="prompt">&#8203;</a>
`.prompt(...)` is the quickest path.
It sends a user message, asks the LLM to respond, and returns the parsed content.
For the default `schema=str`, "parsed content" just means the response text.
A useful beginner rule is:
`.prompt(...)` returns content, not a `Message` object.

In [ ]:
with chats.new("Prompt demo", orphan=True) as prompt_chat:
    prompt_result = agent.prompt("Answer in one sentence: what is a benchmark?")

print("prompt_result:")
print(prompt_result)

print("\nChat messages:")
for message in prompt_chat.messages:
    print("-", message.sender.role, "|", message.text)

prompt_chat

The result is a string.
The chat still contains two messages: the user prompt and the assistant response.
This is helpful when we want a compact one-shot call but still want the trace recorded.

## 4. respond: Answer From The Active Chat <a class="anchor" id="respond" name="respond">&#8203;</a>
`.respond(...)` is the more explicit path.
We add messages ourselves, then ask the LLM to answer from the active chat.
A useful beginner rule is:
`.respond(...)` returns a `Message` object.
That message carries the text, sender, status, reasoning traces, usage metadata, and other model response details.

In [ ]:
with chats.new("Respond demo", orphan=True) as respond_chat:
    actors.user.send("We are comparing two model answers.")
    actors.user.send("Answer briefly: why does the chat history matter?")
    response_message = agent.respond()

print("Returned object type:", type(response_message).__name__)
print("Returned text:", response_message.text)
print("Returned sender:", response_message.sender.name)
print("Returned status:", response_message.status)

print("\nChat message count:", len(respond_chat.messages))

respond_chat

We now have the main difference between the two methods.
Use `.prompt(...)` when we want a short one-shot call and only need the content.
Use `.respond(...)` when the chat already has context or when we want the returned message metadata.

## 5. What Gets Sent To The LLM? <a class="anchor" id="what-gets-sent" name="what-gets-sent">&#8203;</a>
`LLMChat` does not own conversation history.
It reads the current chat from the active context.
More specifically, `.respond(...)` reads direct messages from `chat.messages` and filters out messages where `is_visible_to_llm` is false.
This has two practical effects:
- hidden messages stay in the trace but are not sent to the provider,
- nested chat internals are not flattened into the parent LLM call.

First, let's see hidden messages.
The hidden message remains in the chat, but the LLM only receives visible messages.

In [ ]:
agent.invocations.clear()

with chats.new("Visibility for LLM calls", orphan=True) as visibility_chat:
    actors.user.send("Visible request: summarize the task.")
    actors.system.send("Hidden scoring note: expected answer contains the word trace.", is_visible_to_llm=False)
    message = agent.respond()

print("LLM response:", message.text)

print("\nAll chat messages:")
for item in visibility_chat.messages:
    print("-", item.sender.role, "| visible:", item.is_visible_to_llm, "|", item.text)

print("\nMessages received by invoke():")
for item in agent.invocations[-1]["messages"]:
    print("-", item["role"], "|", item["text"])

visibility_chat

Good, the hidden scoring note stayed out of the provider input.
This is useful for assertions, grading notes, and internal bookkeeping.
It is still metadata behavior, not a security boundary by itself.

Now let's see nested chats.
A parent chat can contain a child chat in `history`, but `respond(...)` reads direct parent messages only.

In [ ]:
agent.invocations.clear()

with chats.new("Outer LLM chat", orphan=True) as outer_chat:
    actors.user.send("Outer question: what should the final answer mention?")

    with chats.new("Inner scratchpad") as inner_chat:
        actors.user.send("Inner note: include private scratch work.")
        agent.respond()

    actors.user.send("Outer follow-up: answer from the outer chat only.")
    outer_response = agent.respond()

print("Outer response:", outer_response.text)

print("\nOuter history item types:")
print([type(item).__name__ for item in outer_chat.history])

print("\nMessages received by the final outer invoke():")
for item in agent.invocations[-1]["messages"]:
    print("-", item["role"], "|", item["text"])

outer_chat

The final outer LLM call did not receive the inner scratchpad messages.
It received only direct messages from the outer chat.
This is one of the most important design choices in the framework.
Nested chats let us keep a full trace without automatically feeding every internal step back into the next model call.

## 6. System Instructions <a class="anchor" id="system-instructions" name="system-instructions">&#8203;</a>
System instructions can reach an LLM in two common ways.
First, a chat can start with `system_instructions`.
In Kaggle Benchmarks, that creates a system message in the chat.
Second, `.respond(system=...)` can pass an extra provider-level system instruction.
Different providers serialize these details differently, but the high-level pattern is the same: instructions guide how the model should answer.

In [ ]:
agent.invocations.clear()

with chats.new(
    "Instruction demo",
    system_instructions="Answer like a careful benchmark assistant.",
    orphan=True,
) as instruction_chat:
    actors.user.send("What are we testing?")
    instruction_message = agent.respond(system="Keep the answer short.")

print("Response:", instruction_message.text)

print("\nChat messages:")
for message in instruction_chat.messages:
    print("-", message.sender.role, "|", message.text)

print("\nProvider-level system argument:")
print(agent.invocations[-1]["system"])

The chat-level system instruction is part of `chat.messages`.
The provider-level `system` argument is passed separately to `.invoke(...)`.
When reading real provider code, this distinction helps explain why system instructions may appear in different request fields.

## 7. Provider Responses And Metadata <a class="anchor" id="metadata" name="metadata">&#8203;</a>
A provider response is more than text.
The framework stores response metadata on the message so benchmark code can inspect it later.
Common fields include:
- `message.text`: readable answer text
- `message.content`: parsed content
- `message.reasoning_traces`: reasoning text when the provider returns it
- `message.tool_calls`: tool call metadata when the provider returns tool calls
- `message.usage`: token, cost, and latency metadata
`LLMResponse` is the small object provider classes return before `.respond(...)` turns it into a framework `Message`.

In [ ]:
with chats.new("Metadata demo", orphan=True) as metadata_chat:
    actors.user.send("Answer briefly: what is metadata?")
    metadata_message = agent.respond(reasoning="low")

print("Text:")
print(metadata_message.text)

print("\nReasoning traces:")
print(metadata_message.reasoning_traces)

print("\nUsage:")
print(metadata_message.usage)

print("\nRaw metadata dictionary:")
print(metadata_message._meta)

The exact fields depend on the provider and model.
Some models return token counts, some return cost fields, and some return reasoning traces only when requested.
The reading pattern stays the same: inspect the response message.

Chat usage can also be aggregated across assistant messages.
This is useful when a benchmark needs to estimate total input and output tokens for a run.

In [ ]:
with chats.new("Usage aggregation demo", orphan=True) as usage_chat:
    agent.prompt("First short question.")
    agent.prompt("Second short question.")

print("Aggregated chat usage:")
print(usage_chat.usage)

Small win: we now know where usage lives on one message and where to read the total for a chat.

## 8. Streaming Responses <a class="anchor" id="streaming" name="streaming">&#8203;</a>
Some providers can return a response in chunks.
In Kaggle Benchmarks, the provider can return an iterator of `LLMResponse` chunks.
The final chat message grows as the chunks arrive.
For learning, we will create a streaming version of our local LLM.

In [ ]:
class StreamingNotebookLLM(NotebookLLM):
    def invoke(self, messages, system=None, reasoning=None, **kwargs):
        if not self.stream_responses:
            return super().invoke(messages, system=system, reasoning=reasoning, **kwargs)

        self.invocations.append(
            {
                "messages": [
                    {
                        "sender": message.sender.name,
                        "role": message.sender.role,
                        "text": message.text,
                    }
                    for message in messages
                ],
                "system": system,
                "reasoning": reasoning,
                "kwargs": kwargs,
            }
        )

        def chunks():
            yield "stream"
            yield "ing "
            yield "works"

        return chunks()


streaming_agent = StreamingNotebookLLM(name="Streaming Notebook LLM")
streaming_agent.stream_responses = True

with chats.new("Streaming demo", orphan=True) as streaming_chat:
    streamed_content = streaming_agent.prompt("Please stream a tiny answer.")

last_streamed_message = streaming_chat.messages[-1]

print("prompt returned:", streamed_content)
print("message content:", last_streamed_message.content)
print("message status:", last_streamed_message.status)
print("message usage:")
print(last_streamed_message.usage)

The final content is one message even though it arrived in chunks.
The metadata reflects the final chunk values because token counts are usually reported as totals, not something to add for each chunk.

## 9. Kaggle ModelProxy <a class="anchor" id="model-proxy" name="model-proxy">&#8203;</a>
In real Kaggle benchmark code, the common entry point is `ModelProxy`.
`ModelProxy(...)` is a factory.
Depending on the `api` argument, it returns an OpenAI-compatible `LLMChat` or a Google GenAI `LLMChat`.
After construction, we still use the same methods we just practiced: `.prompt(...)`, `.respond(...)`, and `.send(...)`.
Model Proxy normally needs two environment variables:
- `MODEL_PROXY_URL`
- `MODEL_PROXY_API_KEY`
We will check for them before trying a real call so this notebook remains runnable without credentials.

In [ ]:
import os

from kaggle_benchmarks.kaggle.model_proxy import ModelProxy, validate_model_proxy_config

model_proxy_ready = bool(os.getenv("MODEL_PROXY_URL") and os.getenv("MODEL_PROXY_API_KEY"))

print("MODEL_PROXY_URL configured:", bool(os.getenv("MODEL_PROXY_URL")))
print("MODEL_PROXY_API_KEY configured:", bool(os.getenv("MODEL_PROXY_API_KEY")))

if model_proxy_ready:
    print("Model Proxy is configured. A real call can run in the next cell.")
else:
    validate_model_proxy_config(
        url=os.getenv("MODEL_PROXY_URL"),
        api_key=os.getenv("MODEL_PROXY_API_KEY"),
        raise_on_error=False,
    )
    print("Skipping real model calls until Model Proxy credentials are configured.")

In [ ]:
print("List available models", os.getenv("LLMS_AVAILABLE"))

The next cell is guarded.
If credentials are present, it constructs a real provider-backed actor and asks one short question.
If credentials are missing, it prints a skip message.
Notice that the code shape is almost identical to our local `NotebookLLM` examples.

In [ ]:
if model_proxy_ready:
    real_agent = ModelProxy(
        model="google/gemma-4-31b",
        api="openai",
        name="Gemma",
    )

    with chats.new("Real ModelProxy demo", orphan=True) as real_chat:
        real_answer = real_agent.prompt("Answer in one short sentence: what is Kaggle?")

    print(real_answer)
    print("Concrete class:", type(real_agent).__name__)
    print("Chat messages:", [message.text for message in real_chat.messages])
else:
    print("Skipped: configure MODEL_PROXY_URL and MODEL_PROXY_API_KEY to run this cell.")

real_chat

The important lesson is not the specific model name.
The important lesson is that `ModelProxy` returns an `LLMChat`-style actor.
Once the actor exists, provider details mostly disappear from the benchmark flow.

## 10. Common Gotchas <a class="anchor" id="common-gotchas" name="common-gotchas">&#8203;</a>
These are the mistakes that usually cause confusion when reading LLM integration code:
- Thinking the LLM object stores conversation history. The active chat stores history.
- Forgetting that `.prompt(...)` returns content, while `.respond(...)` returns a message.
- Expecting `.respond(...)` to read nested chat internals. It reads direct visible messages only.
- Forgetting that `is_visible_to_llm=False` filters a message out of provider input.
- Calling `ModelProxy(...)` before `MODEL_PROXY_URL` and `MODEL_PROXY_API_KEY` are configured.
- Expecting reasoning traces to always exist. They depend on the provider, model, settings, and streaming mode.
- Assuming every provider supports the same temperature, schema, reasoning, or tool-call options.
- Printing or storing real API keys while debugging. Check that keys exist, but do not display them.

One more practical reminder:
Structured outputs use the same `.prompt(...)` and `.respond(...)` methods through the `schema=` argument.
We will study that carefully in Module 3.
For this module, it is enough to know that schema instructions are added around the LLM call, then the response is parsed before being stored as message content.

## 11. Practice Example <a class="anchor" id="practice-example" name="practice-example">&#8203;</a>
This final example combines the module.
We will use `.prompt(...)`, `.respond(...)`, hidden messages, a nested scratchpad, reasoning metadata, and chat usage.
Before reading the output, predict what the final LLM call receives:
- the visible direct messages in the main chat,
- not the hidden scoring note,
- not the inner scratchpad messages.

In [ ]:
practice_agent = NotebookLLM(name="Practice LLM")
practice_agent.invocations.clear()

with chats.new("Module 2 practice", orphan=True) as main_chat:
    first_answer = practice_agent.prompt("Answer briefly: what is Kaggle Benchmarks?")

    actors.system.send(
        "Hidden note for the evaluator: look for the word benchmark.",
        is_visible_to_llm=False,
    )

    with chats.new("Practice scratchpad") as scratchpad_chat:
        actors.user.send("Internal scratchpad note: this should stay inside the child chat.")
        practice_agent.respond()

    actors.user.send("Now answer from the main chat only: what did you see most recently?")
    final_message = practice_agent.respond(reasoning="low")

print("prompt returned:")
print(first_answer)

print("\nrespond returned message:")
print(final_message.text)

print("\nfinal reasoning:")
print(final_message.reasoning_traces)

print("\nmain chat direct messages:")
for message in main_chat.messages:
    print("-", message.sender.role, "| visible:", message.is_visible_to_llm, "|", message.text)

print("\nscratchpad direct messages:")
for message in scratchpad_chat.messages:
    print("-", message.sender.role, "|", message.text)

print("\nmessages received by the final invoke():")
for item in practice_agent.invocations[-1]["messages"]:
    print("-", item["role"], "|", item["text"])

print("\nmain chat usage:")
print(main_chat.usage)

What to observe:
- `.prompt(...)` sent a user message and returned response content.
- `.respond(...)` returned a response message with metadata.
- The hidden system note stayed in the main chat but was not sent to `.invoke(...)`.
- The scratchpad stayed visible in the trace as a nested chat, but its direct messages were not sent to the final main-chat LLM call.
- Usage metadata was available on response messages and aggregated at the chat level.
If those observations make sense, the LLM integration layer is much easier to read.

## 12. Recap <a class="anchor" id="recap" name="recap">&#8203;</a>
`LLMChat` is the bridge between chat history and model providers.
We can use this reading guide:
- **`LLMChat`** means an assistant-style actor that can call a model.
- **`.prompt(...)`** means send a user message, respond, and return content.
- **`.respond(...)`** means read the active chat, respond, and return a message.
- **`.invoke(...)`** means provider-specific work that receives messages and returns an `LLMResponse`.
- **`LLMResponse`** means raw provider output before it becomes a chat message.
- **`ModelProxy`** means the usual Kaggle factory for real provider-backed `LLMChat` actors.
This module builds directly on Module 1.
The model call is new, but the transcript is still built from actors, messages, chats, and contexts.
In the next module, we will make model outputs more predictable with structured schemas.

## Continue Learning

Continue to Module 3: [Structured Outputs](https://www.kaggle.com/code/anhoangvo/kaggle-benchmarks-structured-outputs).

Module 3 shows how to turn model text into predictable Python data. You will use schemas, Pydantic models, dataclasses, parsing errors, and retry patterns so benchmark code can rely on typed outputs.